In [2]:
# ── Cell 1: Load, Filter, Prepare ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

df_raw = pd.read_csv("spatial_seer_all_rooms_v3.csv")

# ── Exclude rescan_num=2 entirely ─────────────────────────────────────────────
df_raw = df_raw[df_raw["rescan_num"] != 2].reset_index(drop=True)

# ── Identify qualifying locations ─────────────────────────────────────────────
# Must have >= 6 scans at rescan_num=0 AND >= 2 scans at rescan_num=1
scan_counts = (
    df_raw.drop_duplicates("scan_id")
    .groupby(["location", "rescan_num"])
    .size()
    .rename("n_scans")
    .reset_index()
)

has_baseline = set(
    scan_counts[(scan_counts["rescan_num"] == 0) & (scan_counts["n_scans"] >= 6)]["location"]
)
has_rescan = set(
    scan_counts[(scan_counts["rescan_num"] == 1) & (scan_counts["n_scans"] >= 2)]["location"]
)
qualifying_locs = sorted(has_baseline & has_rescan)

print(f"Qualifying locations ({len(qualifying_locs)}):")
for loc in qualifying_locs:
    print(f"  {loc:<35} room_type={df_raw[df_raw['location']==loc].iloc[0]['room_label']}")

df = df_raw[df_raw["location"].isin(qualifying_locs)].reset_index(drop=True)

# ── Encoders ──────────────────────────────────────────────────────────────────
loc_enc     = LabelEncoder().fit(df["location"].unique())
room_enc    = LabelEncoder().fit(df["room_label"].unique())
loc_to_room = df.drop_duplicates("location").set_index("location")["room_label"].to_dict()

CHANNELS = [
    "GpuUtil", "CpuUtil", "FrameTimeStdDev", "WorstFrameMs",
    "MainThreadMs", "TotalUsedMem", "CpuClockFreq",
]

SERIES_LEN = df.groupby("scan_id").size().min()

# ── Build scan-level metadata ─────────────────────────────────────────────────
scan_meta = pd.DataFrame([
    {
        "scan_id":    scan_id,
        "location":   group.iloc[0]["location"],
        "room_label": group.iloc[0]["room_label"],
        "rescan_num": int(group.iloc[0]["rescan_num"]),
    }
    for scan_id, group in df.groupby("scan_id")
])
scan_meta["group_key"] = scan_meta["location"] + "__r" + scan_meta["rescan_num"].astype(str)


def build_scan_array(data: pd.DataFrame):
    """Return X (n_scans, n_channels, timesteps), location list."""
    records = []
    for scan_id, group in data.groupby("scan_id"):
        group = group.sort_values("Timestamp").iloc[:SERIES_LEN]
        records.append({
            "location": group.iloc[0]["location"],
            "ts":       group[CHANNELS].values.T,
        })
    X    = np.stack([r["ts"] for r in records])
    locs = [r["location"] for r in records]
    return X, locs


X_all, all_locs = build_scan_array(df)
y_room_all  = room_enc.transform([loc_to_room[l] for l in all_locs])
y_loc_all   = loc_enc.transform(all_locs)
group_keys  = scan_meta["group_key"].values
locations   = np.array(all_locs)

print(f"\nFiltered scans : {len(scan_meta)}")
print(f"Locations      : {len(qualifying_locs)}")
print(f"Room types     : {list(room_enc.classes_)}")
print(f"Series length  : {SERIES_LEN}")
print(f"X shape        : {X_all.shape}")
print(f"Group keys     : {scan_meta['group_key'].nunique()}")


# ── Shared styling helper ─────────────────────────────────────────────────────
def style_cm(disp, ax, title):
    """White cells for zeros, white text on dark cells."""
    ax.set_title(title, fontsize=13, fontweight="bold", pad=15)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    images = ax.get_images()
    norm   = images[0].norm
    cmap_  = images[0].get_cmap()
    for i, row in enumerate(disp.text_):
        for j, text in enumerate(row):
            val = disp.confusion_matrix[i, j]
            if val == 0:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1, color="white", zorder=2))
                text.set_color("black")
            else:
                r, g, b, _ = cmap_(norm(val))
                text.set_color("white" if 0.299*r + 0.587*g + 0.114*b < 0.5 else "black")
            text.set_fontsize(14)
            text.set_fontweight("bold")
            text.set_zorder(3)


print("\nCell 1 complete.")

FileNotFoundError: [Errno 2] No such file or directory: 'spatial_seer_all_rooms_v3.csv'